### OpenRouter API Key

This notebook uses OpenRouter through LangChain's OpenAI-compatible interface. Enter your OpenRouter API key when prompted. You do not need a separate OpenAI API key.


In [ ]:
# OPTIONAL: Install
# pip install -qU langchain langchain-openai langchain-community langchain-pinecone pinecone python-dotenv tiktoken


## Tutorial: Multi‑Step Chain with Retrieval (Summary + Q&A)
We’ll optionally retrieve context from Pinecone, summarize it, then answer the question from the summary.


In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (hidden): ")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model=MODEL, api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1", model=MODEL, temperature=0, seed=42)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
INDEX_NAME = os.getenv("PINECONE_INDEX", "lc-demo-index")
pc = Pinecone(api_key=os.environ.get("PINECONE_API_KEY"))
vectorstore = PineconeVectorStore(index_name=INDEX_NAME, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


### Step 1: Define subchain prompts
- `summary_chain`: compresses context into 3 bullets
- `qa_chain`: answers from the summary only
- (Optional) `retriever`: pulls top‑k docs to form the context if none is provided


In [ ]:
summary_prompt = PromptTemplate.from_template(
    "Summarize the following text into 3 bullet points:\n{context}\n\nBullets:"
)
qa_prompt = PromptTemplate.from_template(
    "Using ONLY the SUMMARY below, answer the QUESTION.\n"
    "If not answerable, say: I don't know.\n\n"
    "SUMMARY:\n{summary}\n\n"
    "QUESTION: {question}\n"
    "ANSWER:"
)

summary_chain = LLMChain(llm=llm, prompt=summary_prompt)
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


### Step 2: Compose multi-step pipeline
No agents: deterministic steps for speed and testability.


In [ ]:
def summarize_then_answer(question: str, context: str = "") -> str:
    if not context:
        # Optional: pull from Pinecone when context not provided
        docs = retriever.get_relevant_documents(question)
        context = "\n\n".join(d.page_content for d in docs)
    summary = summary_chain.run({"context": context})
    answer = qa_chain.run({"summary": summary, "question": question})
    return answer

kb = (
    "LangChain is a framework for building with LLMs. It offers prompts, chains, tools, "
    "and agents to compose complex workflows and integrate with data sources."
)
print(summarize_then_answer("Name two abstractions LangChain provides.", context=kb))
print(summarize_then_answer("What enables retrieval in this setup?"))
